# Streaming Platform Engagement & Churn Simulator
## Exploratory Analysis & Insights

**Subject**: Stochastic Processes and Applications  
**Model**: Markov Chains with Optional Poisson Processes  
**Date**: 2024

This notebook provides comprehensive exploratory analysis of the streaming platform user behavior model.

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sys
from pathlib import Path

# Add src to path
sys.path.insert(0, str(Path().cwd().parent))

from src.markov_model import MarkovChainModel, DEFAULT_TRANSITION_MATRIX
from src.simulation import SimulationEngine, EnhancedSimulationEngine, PoissonArrivalProcess
from src.visualization import MarkovChainVisualizer, SimulationVisualizer

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)
%matplotlib inline

print("✓ Libraries imported successfully")

---
## 1. Initialize Markov Chain Model

In [ ]:
# Create Markov chain with default transition matrix
markov_chain = MarkovChainModel(DEFAULT_TRANSITION_MATRIX)

# Display state names
print("States:")
for i, state in enumerate(markov_chain.state_names):
    print(f"  {i}: {state}")

In [ ]:
# Validate the transition matrix
markov_chain.validate_matrix()

# Display transition matrix as DataFrame
print("\nTransition Probability Matrix:")
print(markov_chain.get_transition_dataframe().round(3))

---
## 2. Markov Chain Properties Analysis

In [ ]:
# Check ergodicity properties
ergodicity = markov_chain.check_ergodicity()

print("\nErgodicity Analysis:")
print(f"  Is Irreducible: {ergodicity['is_irreducible']}")
print(f"  Is Aperiodic: {ergodicity['is_aperiodic']}")
print(f"  Has Unique Steady State: {ergodicity['has_unique_steady_state']}")
print(f"  Is Ergodic: {ergodicity['is_ergodic']}")

if ergodicity['is_ergodic']:
    print("\n✓ This is an ERGODIC Markov chain!")
    print("  → Unique steady-state exists")
    print("  → Independent of initial conditions")
else:
    print("\n⚠ This chain may not be fully ergodic")

---
## 3. Steady-State Analysis

In [ ]:
# Compute steady-state distribution
steady_state = markov_chain.compute_steady_state(method='power')

# Display results
print("\nSteady-State Distribution (π):")
steady_df = pd.DataFrame({
    'State': markov_chain.state_names,
    'Probability': steady_state,
    'Percentage (%)': steady_state * 100
})
print(steady_df.to_string(index=False))

# Verify π = π·P
verification = steady_state @ markov_chain.transition_matrix
print(f"\nVerification (π·P = π):")
print(f"  Max difference: {np.max(np.abs(steady_state - verification)):.2e}")
print(f"  ✓ Verified: {np.allclose(steady_state, verification)}")

In [ ]:
# Visualize steady-state distribution
visualizer = MarkovChainVisualizer(markov_chain)
fig = visualizer.plot_steady_state_distribution(figsize=(12, 6))
plt.tight_layout()
plt.show()

print("\nInterpretation:")
print(f"  Long-run proportion of subscribers: {steady_state[4]*100:.2f}%")
print(f"  Long-run proportion churned: {steady_state[5]*100:.2f}%")

---
## 4. Transition Matrix Visualization

In [ ]:
# Transition matrix heatmap
fig = visualizer.plot_transition_matrix(figsize=(10, 8))
plt.tight_layout()
plt.show()

In [ ]:
# Transition network graph
fig = visualizer.plot_transition_network(figsize=(12, 10), prob_threshold=0.05)
plt.tight_layout()
plt.show()

print("Note: Edge width represents transition probability strength")

---
## 5. Single User Trajectory Simulation

In [ ]:
# Simulate trajectories for multiple users
np.random.seed(42)

n_trajectories = 5
n_steps = 50

trajectories = []
for _ in range(n_trajectories):
    traj = markov_chain.simulate_user_trajectory(n_steps, initial_state_idx=0)
    trajectories.append(traj)

# Display first trajectory
print(f"Sample User Trajectory (states: 0-{len(markov_chain.state_names)-1}):")
print(f"Initial state: {markov_chain.state_names[trajectories[0][0]]}")
print(f"Trajectory: {' → '.join([str(s) for s in trajectories[0][:10]])} ...")
print(f"Final state: {markov_chain.state_names[trajectories[0][-1]]}")

In [ ]:
# Plot trajectory evolution
fig, ax = plt.subplots(figsize=(14, 6))

for i, traj in enumerate(trajectories):
    ax.plot(traj, marker='o', label=f'User {i+1}', linewidth=2, markersize=4, alpha=0.7)

ax.set_yticks(range(len(markov_chain.state_names)))
ax.set_yticklabels(markov_chain.state_names)
ax.set_xlabel('Time Step', fontsize=12, fontweight='bold')
ax.set_ylabel('State', fontsize=12, fontweight='bold')
ax.set_title('Sample User Trajectories Through States', fontsize=14, fontweight='bold')
ax.legend(loc='best')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 6. Monte Carlo Simulation - Base Case

In [ ]:
# Run Monte Carlo simulation
np.random.seed(42)

sim_engine = SimulationEngine(markov_chain)
result = sim_engine.run_simulation(
    n_users=1000,
    n_steps=50,
    seed=42
)

print("\n✓ Simulation completed!")
print(f"  Users simulated: {result.n_users}")
print(f"  Time steps: {result.n_steps}")

In [ ]:
# Display key metrics
metrics = result.get_metrics()

print("\nSimulation Metrics:")
for key, value in metrics.items():
    if isinstance(value, float):
        print(f"  {key}: {value:.2f}")
    else:
        print(f"  {key}: {value}")

In [ ]:
# Visualize results
sim_visualizer = SimulationVisualizer(result.state_names)

# State distribution over time
fig = sim_visualizer.plot_state_distribution_over_time(result, figsize=(14, 7))
plt.tight_layout()
plt.show()

print("\nVisualization: Total users in each state over time")

In [ ]:
# State proportions over time
fig = sim_visualizer.plot_state_proportions_over_time(result, figsize=(14, 7))
plt.tight_layout()
plt.show()

print("\nVisualization: Proportion of users in each state (100% stacked)")

In [ ]:
# Funnel chart
fig = sim_visualizer.plot_funnel_chart(result, figsize=(10, 8))
plt.tight_layout()
plt.show()

print("\nVisualization: User funnel from Visitor to Subscriber")

In [ ]:
# Conversion metrics
fig = sim_visualizer.plot_conversion_metrics(result, figsize=(14, 6))
plt.tight_layout()
plt.show()

---
## 7. Poisson Arrival Process

In [ ]:
# Simulate Poisson arrivals
poisson_process = PoissonArrivalProcess(lambda_rate=15)  # 15 new users per step

arrivals = poisson_process.generate_arrivals(time_horizon=50, seed=42)
cumulative_arrivals = poisson_process.cumulative_arrivals(arrivals)

print(f"Poisson Process (λ = 15):")
print(f"  Total arrivals: {arrivals.sum()}")
print(f"  Mean arrivals per step: {arrivals.mean():.2f}")
print(f"  Std dev: {arrivals.std():.2f}")
print(f"  Min: {arrivals.min()}, Max: {arrivals.max()}")

In [ ]:
# Visualize Poisson arrivals
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Arrivals per step
axes[0].bar(range(len(arrivals)), arrivals, color='#4ECDC4', edgecolor='black', linewidth=1.5)
axes[0].axhline(y=arrivals.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {arrivals.mean():.1f}')
axes[0].set_xlabel('Time Step', fontsize=11, fontweight='bold')
axes[0].set_ylabel('New Arrivals', fontsize=11, fontweight='bold')
axes[0].set_title('Poisson Arrivals per Time Step (λ=15)', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# Cumulative arrivals
axes[1].plot(cumulative_arrivals, marker='o', color='#45B7D1', linewidth=2.5, markersize=5)
axes[1].fill_between(range(len(cumulative_arrivals)), 0, cumulative_arrivals, alpha=0.3, color='#45B7D1')
axes[1].set_xlabel('Time Step', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Cumulative Arrivals', fontsize=11, fontweight='bold')
axes[1].set_title('Cumulative New Users', fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Run simulation with Poisson arrivals
np.random.seed(42)

enhanced_engine = EnhancedSimulationEngine(markov_chain)
result_arrivals, arrivals_data = enhanced_engine.run_simulation_with_arrivals(
    lambda_rate=15,
    n_steps=50,
    seed=42
)

print("\n✓ Simulation with Poisson arrivals completed!")
print(f"  Initial total users: {arrivals_data.sum()}")
print(f"  Final total users: {result_arrivals.n_users}")

In [ ]:
# Compare with and without arrivals
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Without arrivals
df_base = result.get_dataframe()
axes[0].stackplot(
    df_base.index,
    [df_base[col].values for col in result.state_names],
    labels=result.state_names,
    colors=['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A', '#98D8C8', '#C7CEEA'],
    alpha=0.8
)
axes[0].set_xlabel('Time Step', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Number of Users', fontsize=11, fontweight='bold')
axes[0].set_title('Without Poisson Arrivals', fontsize=12, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# With arrivals
df_arrivals = result_arrivals.get_dataframe()
axes[1].stackplot(
    df_arrivals.index,
    [df_arrivals[col].values for col in result_arrivals.state_names],
    labels=result_arrivals.state_names,
    colors=['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A', '#98D8C8', '#C7CEEA'],
    alpha=0.8
)
axes[1].set_xlabel('Time Step', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Number of Users', fontsize=11, fontweight='bold')
axes[1].set_title('With Poisson Arrivals (λ=15)', fontsize=12, fontweight='bold')
axes[1].legend(loc='upper left', fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 8. Sensitivity Analysis

In [ ]:
# Sensitivity analysis: Impact of increasing visitor->subscriber probability
param_values = np.arange(0.0, 0.31, 0.05)

sensitivity_results = sim_engine.run_sensitivity_analysis(
    n_users=1000,
    n_steps=50,
    parameter="Visitor→Subscriber Probability",
    param_values=param_values,
    from_state="Visitor",
    to_state="Subscriber",
    n_simulations=3
)

# Extract results
conversion_rates = [sensitivity_results[pv]['subscriber_conversion_rate'] for pv in param_values]
churn_rates = [sensitivity_results[pv]['churn_rate'] for pv in param_values]
subscribers = [sensitivity_results[pv]['final_subscribers'] for pv in param_values]

print("\nSensitivity Analysis Results:")
sens_df = pd.DataFrame({
    'Visitor→Subscriber Prob': param_values,
    'Conversion Rate (%)': conversion_rates,
    'Churn Rate (%)': churn_rates,
    'Final Subscribers': subscribers
})
print(sens_df.to_string(index=False))

In [ ]:
# Visualize sensitivity analysis
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Conversion rate
axes[0].plot(param_values, conversion_rates, marker='o', linewidth=2.5, markersize=8, color='#45B7D1')
axes[0].set_xlabel('Visitor→Subscriber Probability', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Conversion Rate (%)', fontsize=11, fontweight='bold')
axes[0].set_title('Impact on Conversion Rate', fontsize=12, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Churn rate
axes[1].plot(param_values, churn_rates, marker='s', linewidth=2.5, markersize=8, color='#FF6B6B')
axes[1].set_xlabel('Visitor→Subscriber Probability', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Churn Rate (%)', fontsize=11, fontweight='bold')
axes[1].set_title('Impact on Churn Rate', fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3)

# Subscribers
axes[2].plot(param_values, subscribers, marker='^', linewidth=2.5, markersize=8, color='#98D8C8')
axes[2].set_xlabel('Visitor→Subscriber Probability', fontsize=11, fontweight='bold')
axes[2].set_ylabel('Final Subscribers', fontsize=11, fontweight='bold')
axes[2].set_title('Impact on Subscriber Count', fontsize=12, fontweight='bold')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nSensitivity Insights:")
print(f"  Increasing V→S from {param_values[0]:.2f} to {param_values[-1]:.2f}")
print(f"  Conversion rate change: {conversion_rates[0]:.2f}% → {conversion_rates[-1]:.2f}%")
print(f"  Impact: +{conversion_rates[-1] - conversion_rates[0]:.2f} percentage points")

---
## 9. Key Insights & Conclusions

In [ ]:
print("\n" + "="*70)
print("KEY INSIGHTS & CONCLUSIONS")
print("="*70)

print("\n1. MARKOV CHAIN PROPERTIES:")
print(f"   • Chain is Ergodic: {ergodicity['is_ergodic']}")
print(f"   • Unique steady-state exists: {ergodicity['has_unique_steady_state']}")
print(f"   • Steady-state is independent of initial conditions")

print("\n2. STEADY-STATE ANALYSIS:")
print(f"   • Long-run subscriber proportion: {steady_state[4]*100:.2f}%")
print(f"   • Long-run churn proportion: {steady_state[5]*100:.2f}%")
print(f"   • This represents equilibrium after many time steps")

print("\n3. SIMULATION RESULTS (Base Case, N=1000):")
print(f"   • Conversion rate: {metrics['subscriber_conversion_rate']:.2f}%")
print(f"   • Churn rate: {metrics['churn_rate']:.2f}%")
print(f"   • Final subscribers: {metrics['final_subscribers']}")
print(f"   • Active users remaining: {metrics['active_users']}")

print("\n4. POISSON ARRIVALS:")
print(f"   • With continuous arrivals (λ=15):")
print(f"   • Total users grew to: {result_arrivals.n_users}")
print(f"   • Maintains user growth over time")

print("\n5. SENSITIVITY FINDINGS:")
print(f"   • Small changes in V→S probability significantly impact conversion")
print(f"   • Max sensitivity: {max(conversion_rates) - min(conversion_rates):.2f}% change")
print(f"   • Key lever for optimization: early-stage engagement")

print("\n6. BUSINESS RECOMMENDATIONS:")
print("   • Focus on converting visitors to subscribers early")
print("   • Implement retention strategies (subscriber state has 75% retention)")
print("   • Use Poisson model for realistic user acquisition planning")
print("   • Monitor steady-state to ensure sustainable equilibrium")

print("\n" + "="*70)